# 6. 卷积神经网络


## 6.1 从全连接层到卷积

> **为什么处理图像不能直接用多层感知机（MLP），而必须发明卷积？**

### 1. MLP 处理图像的“三座大山”

假设我们有一张 $1000 \times 1000$ 像素的彩色图片（3 通道）：
1.  **参数爆炸**：如果隐藏层只有 1000 个神经元，全连接层的权重矩阵大小为 $(1000 \times 1000 \times 3) \times 1000 = 3 \times 10^9$。这需要 **12GB** 显存来存储仅仅一层的权重，根本无法训练。
2.  **空间结构丢失**：MLP 要求把图片展平（Flatten）成一维。这导致像素点 $(i, j)$ 与其邻居 $(i+1, j)$ 的关系，和它与远处像素 $(i+500, j)$ 的关系在模型看来是一样的。
3.  **平移脆弱性**：如果在训练集里猫都在左上角，而测试集里的猫出现在右下角，MLP 必须重新学习一遍“右下角的猫”，它无法自动识别这种位置变化。

---

### 2. 卷积的两个物理学直觉

为了解决上述问题，科学家提出了两个核心假设：

#### 2.1. 平移不变性 (Translation Invariance)

**直觉**：不管“猫”出现在图片的哪个位置，它的特征（如尖耳朵、胡须）应该是相同的。

**数学结论**：我们的“特征检测器”在图片的任何地方都应该是**同一套权重**。

#### 2.2. 局部性 (Locality)

**直觉**：要识别一个“鼻子”，我只需要看鼻子周围的像素，不需要看图片另一角的“尾巴”。

**数学结论**：检测器每次只需要关注一个**很小的窗口**。

---

### 3. 数学演化：从全连接到卷积

#### 3.1. 全连接层公式
$$h_{i,j} = \sum_{k,l} W_{i,j,k,l} x_{k,l}$$
这里 $W$ 是一个四维张量。每一对输出坐标 $(i, j)$ 都要看遍输入坐标 $(k, l)$。

#### 3.2. 引入“平移不变性”
我们假设权重只取决于输出点 $(i, j)$ 与输入点 $(k, l)$ 的**相对距离** $(a, b)$，其中 $a = i-k, b = j-l$。
$$h_{i,j} = \sum_{a,b} V_{a,b} x_{i+a, j+b}$$
**变化**：$W$ 变成了 $V$。不管输出 $(i, j)$ 在哪，权重 $V$ 是一样的。**参数量从 $O(H^2 W^2)$ 降到了 $O(HW)$。**

#### 3.3. 引入“局部性”
我们假设当相对距离 $a$ 或 $b$ 超过一定范围（比如 3 像素）时，权重 $V_{a,b} = 0$。
$$h_{i,j} = \sum_{a=-\Delta}^{\Delta} \sum_{b=-\Delta}^{\Delta} V_{a,b} x_{i+a, j+b}$$
**变化**：求和范围缩小到一个极小的窗口。**这就是卷积！** $V$ 就是我们说的**卷积核（Kernel）**。

---

### 4. 对比 MLP 和 CNN 的参数量差异。



In [2]:
from torch import nn

def compare_parameter_counts(height: int, weight: int, channel: int) -> None:
    """对比全连接层和卷积层的参数量。
    
    Args:
        height: 图像高度。
        weight: 图像宽度。
        channel: 输入通道数。
    """
    input_dim = height * weight * channel
    hidden_dim = 1000

    # 1. 模拟全连接层
    fc = nn.Linear(input_dim, hidden_dim)
    fc_params = sum(p.numel() for p in fc.parameters())

    # 2. 模拟卷积层 (假设 3 * 3 卷积核，1000个输出通道)
    # 虽然通道数一样，但卷积核是局部共享的
    conv = nn.Conv2d(in_channels=channel, out_channels=hidden_dim, kernel_size=3)
    conv_params = sum(p.numel() for p in conv.parameters())

    print(f"图像尺寸: {height}x{weight}, 通道: {channel}")
    print(f"全连接层参数量: {fc_params:,}")
    print(f"卷积层参数量: {conv_params:,}") # 3 * 3 * 3 * 1000 + 1000
    print(f"参数压缩倍数: {fc_params / conv_params:.2f}x")

# 运行对比 (假设 128x128 的小图)
compare_parameter_counts(128, 128, 3)

图像尺寸: 128x128, 通道: 3
全连接层参数量: 49,153,000
卷积层参数量: 28,000
参数压缩倍数: 1755.46x


**实验结论**：
你会发现，对于 $128 \times 128$ 的图片，MLP 需要近 **5000 万**个参数，而卷积层只需要 **2.8 万**个。卷积层不仅效率高，而且强制模型学习“局部特征”。

---

### 5. 核心术语总结

*   **通道 (Channels)**：图像的维度。输入通常是 RGB（3通道），隐藏层可以有成百上千个通道（每个通道代表一种特征，如颜色、边缘、纹理）。
*   **感受野 (Receptive Field)**：输出中的一个像素能“看”到输入图片的范围。层数越深，感受野越大。
*   **互相关 (Cross-Correlation)**：代码里实际运行的算法，直接把卷积核 $K$ 放在 $X$ 的窗口上，对应位置相乘再求和。
*   **卷积 (Convolution)**：在进行乘法之前，先将卷积核 $K$ 水平翻转，再垂直翻转（相当于旋转 180 度），然后再执行相乘求和。但深度学习中由于核是训练出来的，翻不翻转结果一样，所以通用术语叫卷积。

---


## 6.2 图像卷积

> 从底层实现二维互相关运算，并学习如何通过数据“学习”出一个卷积核。

### 1. 二维互相关运算

在深度学习框架中，“卷积层”实际上实现的是互相关运算。

#### 1.1 运算逻辑
卷积核（Kernel）在输入张量上滑动。在每个位置，输入子张量与卷积核按元素相乘并求和，得到输出张量中的单点数值。

#### 1.2 输出形状公式
若输入形状为 $n_h \times n_w$，卷积核形状为 $k_h \times k_w$，则输出形状为：
$$(n_h - k_h + 1) \times (n_w - k_w + 1)$$

---

### 2. 手动实现卷积算子

我们将编写一个通用的互相关函数，这是所有 CNN 层的数学心脏。


In [3]:
import torch
from torch import nn, Tensor

def corr2d(X: Tensor, K: Tensor) -> Tensor:
    """计算二维互相关运算。
    
    这是卷积层最底层的数学实现。

    Args:
        X: 输入张量，形状为 (h, w)。
        K: 卷积核张量，形状为 (kh, kw)。
    
    Returns:
        Tensor: 输出特征图，形状为 (h - kh + 1, w - kw + 1)。
    """
    h, w = X.shape
    kh, kw = K.shape

    # 1. 根据公式预分配输出空间
    Y = torch.zeros((h - kh + 1, w - kw + 1), device=X.device)

    # 2. 嵌套循环实现滑动窗口
    # 注意：在生产环境（如 nn.Conv2d）中，这部分由高效的 C++/CUDA 算子完成
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            # 核心动作：切片 (Slicing) -> 逐元素相乘 -> 求和
            Y[i, j] = (X[i: i + kh, j: j+kw] * K).sum()

    return Y

# --- 简单验证 ---
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K_test = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
# 输出应为 [[19, 25], [37, 43]]
# 计算过程示例: (0*0 + 1*1 + 3*2 + 4*3) = 19
print(f"验证输出:\n{corr2d(X_test, K_test)}")

验证输出:
tensor([[19., 25.],
        [37., 43.]])


In [4]:
# 扩展

def corr2d_multi_in(X: Tensor, K: Tensor) -> Tensor:
    """处理多输入通道的互相关运算。
    
    Args:
        X: 输入张量，形状为 (c_in, h, w)。
        K: 卷积核张量，形状为 (c_in, kh, kw)。
    
    Returns:
        Tensor: 单通道输出特征图，形状为 (h - kh + 1, w - kw + 1)。
    """

    # 逻辑：对每个输入通道分别进行互相关计算，然后将结果相加
    # zip(X, K) 会配对相应的输入通道和卷积核通道
    # 此时 x 形状为 (h, w)， k 形状为 (kh, kw)。
    # 最后将 c_in 个 2D 特征图叠加为一个 2D 特征图
    return sum(corr2d(x, k) for x, k in zip(X, K))

def corr2d_multi_in_out(X: Tensor, K: Tensor) -> Tensor:
    """处理多输入通道、多输出通道的互相关运算。
    
    Args:
        X: 输入张量，形状为 (c_in, h, w)。
        K: 卷积核张量，形状为 (c_out, c_in, kh, kw)。
    
    Returns:
        Tensor: 多通道输出特征图，形状为 (c_out, h - kh + 1, w - kw + 1)。
    """
    # 逻辑：遍历每个输出通道对应的卷积核，计算多通道输入互相关，最后堆叠结果
    # torch.stack: 将列表中的多个张量，沿着一个新的维度拼在一起。
    # 0 表示在第一维拼接
    # 结果：将 c_out 个形状为 (h_out, w_out) 的 2D 张量，拼成一个形状为 (c_out, h_out, w_out) 的 3D 张量
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

# --- 验证样例 ---

# 1. 构造输入 X: (2个通道, 3x3 图像)
# 通道 0 全为 0, 1, 2...
# 通道 1 比通道 0 各项加 1
X_val = torch.tensor([
    [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
])

# 2. 构造卷积核 K: (2个输出通道, 2个输入通道, 2x2 核)
# 这里为了方便口算，我们设第一个输出通道的核全为 1，第二个输出通道的核全为 0
K_val = torch.stack([
    torch.ones((2, 2, 2)), # 输出通道 0：全 1 核
    torch.zeros((2, 2, 2)) # 输出通道 1：全 0 核
], 0)

# 3. 执行运算
Y_val = corr2d_multi_in_out(X_val, K_val)

# 4. 打印结果
print(f"输入形状: {X_val.shape}")  # (2, 3, 3)
print(f"卷积核形状: {K_val.shape}") # (2, 2, 2, 2)
print(f"输出形状: {Y_val.shape}")  # (2, 2, 2)
print(f"验证输出结果:\n{Y_val}")

输入形状: torch.Size([2, 3, 3])
卷积核形状: torch.Size([2, 2, 2, 2])
输出形状: torch.Size([2, 2, 2])
验证输出结果:
tensor([[[20., 28.],
         [44., 52.]],

        [[ 0.,  0.],
         [ 0.,  0.]]])


---

### 3. 自定义卷积层类


In [5]:
class MyConv2D(nn.Module):
    """自定义二维卷积层模块。"""

    def __init__(self, kernel_size: tuple[int, int]):
        """
        Args:
            kernel_size: 卷积核的形状 (高度, 宽度)。
        """
        super().__init__()
        # 注册可学习参数：权重 (Weight) 和 偏置 (Bias)
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, X: Tensor) -> Tensor:
        """前向传播逻辑。"""
        return corr2d(X, self.weight) + self.bias
    
# 1. 实例化自定义卷积层，传入 2x2 的卷积核尺寸
conv_layer = MyConv2D(kernel_size=(2, 2))

# 2. 准备输入数据
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])

# 3. 执行前向传播
Y_test = conv_layer(X_test)

# 4. 打印结果
print(list(conv_layer.parameters()))
print(f"输入形状: {X_test.shape}")
print(f"输出形状: {Y_test.shape}")
print(f"前向传播结果:\n{Y_test}")

[Parameter containing:
tensor([[0.9874, 0.5555],
        [0.3430, 0.2169]], requires_grad=True), Parameter containing:
tensor([0.], requires_grad=True)]
输入形状: torch.Size([3, 3])
输出形状: torch.Size([2, 2])
前向传播结果:
tensor([[ 2.4521,  4.5549],
        [ 8.7606, 10.8634]], grad_fn=<AddBackward0>)


In [6]:
# 支持输入输出多通道的卷积层

class MyConv2DFull(nn.Module):
    """完整实现二维卷积层模块，支持多输入通道和多输出通道。"""
    def __init__(self, in_channels: int, out_channels: int, kernel_size: tuple[int, int]):
        """
        Args:
            in_channels: 输入通道数。
            out_channels: 输出通道数。
            kernel_size: 卷积核的形状 (高度, 宽度)。
        """
        super().__init__()
        # 1. 权重形状: (输出通道, 输入通道, 核高度, 核宽度)
        self.weight = nn.Parameter(torch.rand(out_channels, in_channels, *kernel_size))

        # 2. 偏置：每个输出通道一个偏置值
        # 形状为 (c_out, 1, 1)
        self.bias = nn.Parameter(torch.zeros(out_channels, 1, 1))
    
    def forward(self, X: Tensor) -> Tensor:
        """
        Args:
            X: 输入张量，形状为 (c_in, h, w)

        Returns:
            Tensor: 输出张量，形状为 (c_out, h_out, w_out)
        """
        return corr2d_multi_in_out(X, self.weight) + self.bias
    
# --- 验证完整卷积层 ---

# 实例化：3个输入通道（如 RGB），16个输出通道，3x3 的核
conv_layer_full = MyConv2DFull(in_channels=3, out_channels=16, kernel_size=(3, 3))

# 随机生成一个 3 通道的图像输入 (3, 32, 32)
X_input = torch.randn(3, 32, 32)

# 执行前向传播
Y_output = conv_layer_full(X_input)

print(f"输入形状: {X_input.shape}")
print(f"权重形状: {conv_layer_full.weight.shape}")
print(f"偏置形状: {conv_layer_full.bias.shape}")
print(f"输出形状: {Y_output.shape}") 
# 预期形状: (16, 32-3+1, 32-3+1) -> (16, 30, 30)

输入形状: torch.Size([3, 32, 32])
权重形状: torch.Size([16, 3, 3, 3])
偏置形状: torch.Size([16, 1, 1])
输出形状: torch.Size([16, 30, 30])


---

### 4. 手工设计边缘检测器

> 卷积核的本质是**特征提取器**。

In [7]:
# 设计一个核来寻找图像的垂直边缘。

def manual_edge_detection() -> None:
    """演示手动设计的卷积核如何提取边缘特征。"""
    # 1. 构造一个 6 x 8 的图像：中间四列为黑 (0)，两边为白 (1)
    X = torch.ones((6, 8))
    X[:, 2:6] = 0
    print("原始图像 (1=白, 0=黑):\n", X)

    # 2. 构造一个 1 x 2 的垂直边缘检测核
    # 逻辑：如果左右像素一致，结果为 0；如果不一致（边缘），结果非 0
    K = torch.tensor([[1.0, -1.0]])

    # 3. 执行运算
    Y = corr2d(X, K)
    print("\n特征图 (检测到的边缘):\n", Y)
    # 结果分析：1 代表从白变黑的边缘，-1 代表从黑变白的边缘

manual_edge_detection()

原始图像 (1=白, 0=黑):
 tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

特征图 (检测到的边缘):
 tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


---

### 5. 从数据学习卷积核

> 深度学习可以通过反向传播让模型自己“悟”出这个核。


In [8]:
def train_cov_kernel() -> None:
    """通过训练学习卷积核参数。"""
    # 1. 准备数据
    X = torch.ones((6, 8))
    X[:, 2:6] = 0
    K_target = torch.tensor([[1.0, -1.0]])
    Y_target = corr2d(X, K_target)

    # 2. 构造内置卷积层 (in_channels=1, out_channels=1, kernel=(1,2))
    # 注意：Conv2d 期待 4D 输入 (Batch, Channel, H, W)
    conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

    # 将数据转换为 4D
    X_4d = X.reshape((1, 1, 6, 8))
    Y_4d = Y_target.reshape((1, 1, 6, 7))

    lr = 3e-2
    print(f">>> 目标核: {K_target}")
    print(f">>> 初始核: {conv2d.weight}")
    print("\n>>> 开始训练学习卷积核...")

    for i in range(20):
        Y_hat = conv2d(X_4d)
        # 使用均方误差 (MSE) 作为损失函数
        loss = (Y_hat - Y_4d) ** 2

        conv2d.zero_grad()
        loss.sum().backward()

        # 正常来讲这里应该要用优化器
        # 不过为了方便演示，这里采用以下写法，手动实现梯度下降。
        with torch.no_grad():
            conv2d.weight -= lr * conv2d.weight.grad

        if (i + 1) % 5 == 0:
            print(f'Epoch {i+1}, Loss: {loss.sum().item():.5f}, 当前核: {conv2d.weight.reshape(1, 2)}')

    # 3. 检查最终结果
    learned_K = conv2d.weight.reshape(1, 2)
    print(f"\n训练结束！学习到的卷积核参数:\n{learned_K}")
    print(f"误差距离: {torch.abs(learned_K - K_target).sum().item():.6f}")


train_cov_kernel()

>>> 目标核: tensor([[ 1., -1.]])
>>> 初始核: Parameter containing:
tensor([[[[-0.0352,  0.0857]]]], requires_grad=True)

>>> 开始训练学习卷积核...
Epoch 5, Loss: 0.38627, 当前核: tensor([[ 0.8779, -0.8944]], grad_fn=<ViewBackward0>)
Epoch 10, Loss: 0.00507, 当前核: tensor([[ 0.9905, -0.9851]], grad_fn=<ViewBackward0>)
Epoch 15, Loss: 0.00012, 当前核: tensor([[ 0.9978, -0.9996]], grad_fn=<ViewBackward0>)
Epoch 20, Loss: 0.00001, 当前核: tensor([[ 1.0002, -0.9996]], grad_fn=<ViewBackward0>)

训练结束！学习到的卷积核参数:
tensor([[ 1.0002, -0.9996]], grad_fn=<ViewBackward0>)
误差距离: 0.000582


---

### 6. 细节补充


1.  **感受野 (Receptive Field)**：
    *   在卷积网络中，对于任一层的任意元素 $x$，其感受野是指在前向传播期间可能影响 $x$ 计算的所有输入。
    *   **教材要点**：我们可以通过堆叠多个小卷积核层来代替一个大卷积核层，这样不仅参数更少，而且能引入更多的非线性激活。
2.  **互相关 vs 卷积**：
    *   数学上的卷积需要将核先进行水平和垂直翻转。
    *   **工程结论**：由于卷积核本身是随机初始化并训练出来的，翻不翻转完全不影响模型的表达能力。为了代码简洁，所有深度学习框架默认实现的都是互相关。
3.  **特征图 / 特征映射 (Feature Map)**：
    *   二维卷积层的输出有时被称为特征图，因为它可以被视为输入在空间维度上提取出的特征。

---


## 6.3 填充和步幅

> 每次卷积后，输出的尺寸都会比输入小一点。如果网络很深，图片很快就会缩减到，且边缘信息会迅速丢失。
>
> 为了解决这个问题，我们需要引入两个关键的超参数：**填充**和**步幅**。


### 1. 填充 (Padding)

#### 1.1 核心动机

*   **保持空间维度**：防止图像在经过多层卷积后变得太小。
*   **保护边缘信息**：让卷积核有机会“正眼看”图像边缘的像素，而不是扫一眼就过去。

#### 1.2 数学定义

在输入图像的四周添加额外的行和列（通常填充值为 0）。
如果添加 $p_h$ 行填充（上下各 $p_h/2$）和 $p_w$ 列填充（左右各 $p_w/2$），输出形状变为：
$$(n_h - k_h + p_h + 1) \times (n_w - k_w + p_w + 1)$$

*   **工程惯例**：通常设置 $p_h = k_h - 1$ 和 $p_w = k_w - 1$，这样当步幅为 1 时，输入和输出的形状完全一致。

---

### 2. 步幅 (Stride)

#### 2.1 核心动机

*   **下采样 (Downsampling)**：大幅度降低图像的宽高 (即缩小图像分辨率)，减少计算量。
*   **增加感受野**：让卷积核以更快的速度扫过整张图，并在一定程度上提高了全局性。

> 补充：实现下采样除了带步幅的卷积之外还可以考虑池化层。

#### 2.2 数学定义

步幅是指卷积核每次滑动的行数和列数。
如果行步幅为 $s_h$，列步幅为 $s_w$，最终的输出形状通用公式为：
$$\lfloor (n_h - k_h + p_h + s_h) / s_h \rfloor \times \lfloor (n_w - k_w + p_w + s_w) / s_w \rfloor$$

---

### 3. 验证形状变换

我们将编写一个辅助函数，用于验证不同参数下的卷积输出形状。



In [9]:
import torch
from torch import nn, Tensor
from typing import Union

def comp_conv2d_output_shape(conv2d: nn.Module, X: Tensor) -> tuple[int, ...]:
    """计算卷积层处理后的输出形状。
    
    该工具函数常用于工程调试，确保模型层之间的维度对接正确。

    Args:
        conv2d: 已经实例化的卷积层。
        X: 输入张量，形状为 (batch_size, channels, h, w)。
    
    Returns:
        tuple[int, ...]: 输出张量形状。
    """
    # 执行一次前向传播
    # 注意：这里不需要计算梯度
    with torch.no_grad():
        Y = conv2d(X)
    return Y.shape

# --- 验证 1: 填充 (Padding) ---
# 创建一个 8x8 的输入
X = torch.randn(size=(1, 1, 8, 8))

# 卷积核 3x3，上下左右各填充 1 行/列。输出应保持 8x8
# 计算: (8 - 3 + 2*1 + 1) = 8
conv_p1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1)
print(f"Padding=1 输出形状: {comp_conv2d_output_shape(conv_p1, X)}")

# --- 验证 2: 步幅 (Stride) ---
# 卷积核 3x3，填充 1，步幅 2。输出应减半为 4x4
# 计算: floor((8 - 3 + 2*1 + 2) / 2) = 4
conv_s2 = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
print(f"Stride=2 输出形状: {comp_conv2d_output_shape(conv_s2, X)}")

# 进阶：不同方向的填充和步幅
# padding=(上下, 左右)
# stride=(纵向, 横向)
conv_special = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
# 输入 8x8 -> 高: floor((8-3+0+3)/3)=2, 宽: floor((8-5+2*1+4)/4)=2
print(f"非对称参数输出形状: {comp_conv2d_output_shape(conv_special, X)}")

Padding=1 输出形状: torch.Size([1, 1, 8, 8])
Stride=2 输出形状: torch.Size([1, 1, 4, 4])
非对称参数输出形状: torch.Size([1, 1, 2, 2])


---

### 4. 细致梳理：本节关键知识点

1.  **填充的默认值**：PyTorch 默认 `padding=0`，`stride=1`。
2.  **为什么填充通常用 0？**：0 填充（Zero Padding）在数学上最简单，且不会给模型引入额外的偏差信息。
3.  **下采样策略**：在现代 CNN（如 ResNet）中，我们通常不使用池化层来减小尺寸，而是直接使用 `stride=2` 的卷积层。
4.  **步幅公式与降维**：宽高缩减 = 像素点变少 = 数学输入变量变少 = 维度降低。（宽高缩减也可称为**下采样**）

---

### 5. 细节补充

*   **“Same” 卷积**：在 Keras 等框架中有 `padding='same'` 选项，PyTorch 1.9+ 也支持了这一写法。它会自动计算填充量，使输出与输入维度一致。

    ```python
    # 自动保持尺寸
    # 注意：在 padding = 'same' 时，强制要求步幅 stride = 1
    conv_same = nn.Conv2d(1, 1, kernel_size=3, padding='same')
    ```

*   **显存陷阱**：虽然步幅变大可以减小输出张量的大小，从而节省显存，但过大的步幅（如 `stride=4`）会导致严重的特征丢失。

---


## 6.4 多输入多输出通道

> 这个好像在前面实现了...

### 1. 多输入通道 (Multiple Input Channels)

当输入包含多个通道时（如 RGB 图像），卷积核也必须具有相同的通道数。

#### 1.1 运算逻辑
1.  为每个输入通道分配一个独立的二维卷积核。
2.  对每个通道分别进行互相关运算。
3.  将所有通道的结果**相加**，得到一个单通道的输出。

#### 1.2 手动实现多通道输入卷积 (跟前面的一样)

```python
import torch
from torch import Tensor

def corr2d_multi_in(X: Tensor, K: Tensor) -> Tensor:
    """处理多输入通道的互相关运算。

    Args:
        X: 输入张量，形状为 (c_in, h, w)。
        K: 卷积核张量，形状为 (c_in, kh, kw)。

    Returns:
        Tensor: 单通道输出特征图，形状为 (h - kh + 1, w - kw + 1)。
    """
    # 沿着 X 和 K 的第 0 维（通道维）遍历并求和
    # corr2d 是前面 6.2 实现的那个。
    return sum(corr2d(x, k) for x, k in zip(X, K))
```

---

### 2. 多输出通道 (Multiple Output Channels)

为了提取多种不同的特征（如一个通道查边缘，一个通道查颜色），我们需要多个输出通道。

#### 2.1 运算逻辑

1.  准备 $c_{out}$ 个不同的多通道卷积核。
2.  每个卷积核产生一个单通道输出。
3.  将这些输出**堆叠**在一起，形成形状为 $(c_{out}, h_{out}, w_{out})$ 的张量。

#### 2.2 手动实现全功能卷积 (依旧是前面实现过的)

```python
def corr2d_multi_in_out(X: Tensor, K: Tensor) -> Tensor:
    """处理多输入通道、多输出通道的互相关运算。

    Args:
        X: 输入张量，形状为 (c_in, h, w)。
        K: 卷积核张量，形状为 (c_out, c_in, kh, kw)。

    Returns:
        Tensor: 多通道输出特征图，形状为 (c_out, h_out, w_out)。
    """
    # 遍历 K 的第 0 维，对每个输出通道的核执行 multi_in 运算
    # 然后在第 0 维进行堆叠 (stack)
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)
```

---

### 3. $1 \times 1$ 卷积层 (The $1 \times 1$ Convolutional Layer)

这是一个极其重要且富有创意的概念，由 *Network in Network* 论文提出，并在 ResNet 和 Inception 中发扬光大。

> $1 \times 1$ 卷积层通常用于调整网络层的通道数量和控制模型复杂性。

#### 3.1 为什么它不是“脱裤子放屁”？
虽然 $1 \times 1$ 卷积在空间上没有跨度，但它在**通道维度**上执行了全连接：
*   **跨通道集成**：它将每个像素位置上的 $c_{in}$ 个值进行加权求和。
*   **降维/增维**：通过控制 $c_{out}$，可以灵活地压缩或扩展通道数，而不改变图像的分辨率。

#### 3.2 验证 $1 \times 1$ 卷积等价于通道维度上的全连接


In [10]:
def corr2d_1x1(X: Tensor, K: Tensor) -> Tensor:
    """使用全连接层的逻辑实现 1x1 卷积。"""
    # 假设输入 X 形状为 (c_in, h, w)
    c_in, h, w = X.shape
    c_out = K.shape[0]

    # 1. 将输入展平为 (通道，像素总数)
    X = X.reshape((c_in, h * w))
    # 2. 将卷积核展平为 (输出通道，输入通道)
    K = K.reshape((c_out, c_in))

    # 3. 矩阵乘法：相当于在每个像素点上做全连接
    Y = K @ X
    return Y.reshape((c_out, h, w))

# --- 验证等价性 ---
X_1x1 = torch.normal(0, 1, (3, 3, 3))
K_1x1 = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out(X_1x1, K_1x1)
Y2 = corr2d_1x1(X_1x1, K_1x1)
print(f"1x1 卷积等价性验证: {torch.allclose(Y1, Y2)}")

1x1 卷积等价性验证: True


---

### 4. 细致梳理

1.  **参数量计算**：

    *   输入通道 $c_{in}$，输出通道 $c_{out}$，核大小 $k_h \times k_w$。
    *   总参数量 = $c_{in} \times c_{out} \times k_h \times k_w$（计偏置再加 $c_{out}$）。

2.  **通道的物理意义**：

    *   在浅层，通道可能代表颜色或简单线条。
    *   在深层，通道代表更复杂的语义特征（如“猫耳朵”、“车轮”）。

3.  **$1 \times 1$ 卷积的妙用**：
    
    *   它是调整模型**计算量**和**参数量**的神器。
    *   示例 (Bottleneck 雏形，常用于高通道数层) (294912 > 77824)：
        
        Conv2d(128, 256, kernel_size=3) -> Conv2d(128, 32, kernel_size=1) + Conv2d(32, 256, kernel_size=3)

1.  **多输入与多输出**：

    *   多输入：所有输入通道的结果最后是**相加**的。
    *   多输出：输出通道数等于**卷积核的个数**。

---

### 5. 工程细节补充

*   **计算瓶颈**：卷积层的计算量很大程度上取决于通道数的乘积 ($c_{in} \times c_{out}$)。
*   **PyTorch 接口**：`nn.Conv2d(in_channels, out_channels, kernel_size)`。
    *   你只需要指定输入和输出通道数，PyTorch 会自动为你创建 4D 形状的权重张量。
*   **Batch Norm**：在现代深度学习中，如果卷积层后面紧跟着**批量归一化层（Batch Normalization）**，我们通常会在定义卷积时设置 `bias=False`。

---


## 6.5 汇聚层/池化层 (Pooling)

> 为了让神经网络对**位置偏移**更具鲁棒性，并进一步降低计算量，我们需要引入**汇聚层**（也常被称为池化层）。

### 1. 核心动机：为什么要“汇聚”？

1.  **平移不变性 (Invariance)**：即使目标物体在图像中发生了微小的位移，汇聚层也能提取出相似的特征。
2.  **下采样 (Downsampling)**：减小特征图的空间维度（宽和高），从而减少后续层的计算量。
3.  **防止过拟合**：通过减少参数数量和提取抽象特征，起到一定的正则化作用。

---

### 2. 汇聚方式：最大汇聚与平均汇聚

> 在现代 CNN 的**隐含层（Internal Layers）** 中，几乎 90% 以上的情况都会首选 **Max Pooling**，因为它能更精准地传递“特征是否存在”的信息。而 **Average Pooling** 现在更多地出现在网络的最后一层（Global Average Pooling），用来整合全图信息并降维。

#### 2.1 最大汇聚 (Max Pooling)
*   **逻辑**：在窗口内取**最大值**。
*   **物理意义**：只要窗口内出现了某个特征（高响应值），就将其保留。它对纹理提取非常有效。

#### 2.2 平均汇聚 (Average Pooling)
*   **逻辑**：在窗口内取**平均值**。
*   **物理意义**：保留窗口内的背景信息或整体特征，输出更平滑。

---

### 3. 手动实现汇聚算子

> 实际上 pooling 可以视为特殊固定的卷积核。

我们将编写一个通用的汇聚函数，支持最大和平均两种模式。


In [11]:
import torch
from torch import Tensor

def pool2d(X: Tensor, pool_size: tuple[int, int], mode: str = 'max') -> Tensor:
    """手动实现二维汇聚运算。
    
    Args:
        X: 输入张量，形状为 (h, w)。
        pool_size: 汇聚窗口的大小 (ph, pw)。
        mode: 汇聚模式，可选 'max' 或 'avg' 。

    Returns:
        Tensor: 汇聚后的特征图。
    """
    h, w = X.shape
    ph, pw = pool_size
    # 输出形状与无填充卷积一致
    Y = torch.zeros((h - ph + 1, w - pw + 1))

    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            # 提取当前窗口区域
            window = X[i: i + ph, j: j + pw]

            if mode == 'max':
                Y[i, j] = window.max()
            elif mode == 'avg':
                Y[i, j] = window.mean()
            else:
                raise ValueError("不支持的汇聚模式，请选择 'max' 或 'avg'")
    return Y

# --- 验证 ---
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
print(f"2x2 最大汇聚结果:\n{pool2d(X, (2, 2), 'max')}")
print(f"2x2 平均汇聚结果:\n{pool2d(X, (2, 2), 'avg')}")

2x2 最大汇聚结果:
tensor([[4., 5.],
        [7., 8.]])
2x2 平均汇聚结果:
tensor([[2., 3.],
        [5., 6.]])


In [12]:
# 支持多通道的汇聚运算
def pool2d_multi_chan(X: Tensor, pool_size: tuple[int, int], mode: str = 'max') -> Tensor:
    """支持多通道输入的二维汇聚运算。
    
    Args:
        X: 输入张量，形状为 (c, h, w)。
        pool_size: 汇聚窗口的大小 (ph, pw)。
        mode: 汇聚模式，可选 'max' 或 'avg' 。

    Returns:
        Tensor: 汇聚后的特征图，形状为 (c, h_out, w_out)。
    """
    # 对每一个通道独立调用我们之前写的 pool2d 函数
    # x 的形状为 (h, w)
    return torch.stack([pool2d(x, pool_size, mode) for x in X], 0)

# --- 验证 ---
# 1. 构造一个 3 通道输入 (3, 3, 3)
# unsqueeze 升维，unsqueeze(0) 在第 0 维插入一个长度为 1 的新轴。
# 实际下面一行代码与 
# X_multi_test = torch.stack([X, X+1, X+2], 0) 等价
X_multi_test = torch.cat([X.unsqueeze(0), (X+1).unsqueeze(0), (X+2).unsqueeze(0)], 0)
print(f"输入形状: {X_multi_test.shape}")

# 2. 执行 2x2 最大汇聚
Y_multi_test = pool2d_multi_chan(X_multi_test, (2, 2), 'max')

print(f"输出形状: {Y_multi_test.shape}") # 预期应为 (3, 2, 2)
print(f"输出结果:\n{Y_multi_test}")

输入形状: torch.Size([3, 3, 3])
输出形状: torch.Size([3, 2, 2])
输出结果:
tensor([[[ 4.,  5.],
         [ 7.,  8.]],

        [[ 5.,  6.],
         [ 8.,  9.]],

        [[ 6.,  7.],
         [ 9., 10.]]])


---

### 4. 填充、步幅与多通道

汇聚层在 PyTorch 官方实现中（如 `nn.MaxPool2d`、`nn.AvgPool2d`）同样支持 `padding` 和 `stride`。

#### 4.1 官方接口演示


In [13]:
# 构造 4D 输入: (Batch, Channel, H, W)
X_4d = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))

# 默认情况下，PyTorch 的步幅 (stride) 等于窗口大小 (kernel_size)，确保窗口之间不堆叠。
# 这与卷积层默认步幅为 1 的行为不同，需要特别注意
pool2d_layer = nn.MaxPool2d(kernel_size=3, padding=1, stride=2)
print(f"官方 MaxPool2d 输出形状: {pool2d_layer(X_4d).shape}")

官方 MaxPool2d 输出形状: torch.Size([1, 1, 2, 2])


#### 4.2 多通道行为 (关键区别！)

> 汇聚层具有通道独立性，即不改变通道数。

这是汇聚层与卷积层最本质的区别：

*   **卷积层**：会将所有输入通道的结果**相加**，输出通道数由卷积核个数决定。
*   **汇聚层**：对每个输入通道**独立运算**。输入有多少通道，输出就有多少通道。


In [14]:
# 构造 3 通道输入
X_multi = torch.cat((X_4d, X_4d + 1, X_4d + 2), 1) # 形状 (1, 3, 4, 4)
pool_multi = nn.MaxPool2d(3, padding=1, stride=2)
print(f"多通道汇聚后形状: {pool_multi(X_multi).shape}") 
# 预期输出: torch.Size([1, 3, 2, 2]) -> 通道数没变！

多通道汇聚后形状: torch.Size([1, 3, 2, 2])


**注意**：

*   `torch.cat` 用以**特征拼接**和**通道拼接**。
*   `torch.stack` 用以**构建 Batch**。

---

### 5. 细致梳理

1.  **无参数性**：汇聚层没有可学习的权重 $W$ 或偏置 $b$。它只是一种固定的数学变换。
2.  **空间鲁棒性**：最大汇聚层能容忍输入特征在 $k_h-1$ 或 $k_w-1$ 范围内的平移。
3.  **常用配置**：
    *   窗口大小通常为 $2 \times 2$ 或 $3 \times 3$。
    *   步幅通常等于窗口大小，以实现无重叠的下采样。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **现代趋势**：在一些现代架构（如近年的 ResNet 变体或 Vision Transformer）中，人们开始倾向于使用 `stride=2` 的卷积层来替代汇聚层。因为卷积层是可学习的，模型可以自己决定如何下采样。
*   **全局平均汇聚 (Global Average Pooling)**：在网络的最后几层，常将窗口大小设为整个特征图的大小（例如 $7 \times 7$ 的图做 $7 \times 7$ 汇聚），将其变成 $1 \times 1$。这常用于替代最后的全连接层，能显著减少参数量并防止过拟合。

---


## 6.6 卷积神经网络 (LeNet)

> 这是深度学习历史上的一个里程碑。LeNet-5 由 Yann LeCun 在 1998 年提出，最初用于手写数字识别（MNIST）。它是第一个成功商用的卷积神经网络，确立了 **“卷积 + 汇聚 -> 全连接”** 的经典架构模式。

### 1. LeNet 的架构设计

LeNet 的设计哲学是：**通过卷积层提取空间特征，通过汇聚层降低敏感度，最后通过全连接层进行非线性分类。**

#### 1.1 结构拆解（从左到右）

1.  **输入层**：$1 \times 28 \times 28$（Fashion-MNIST 尺寸）。
2.  **第一块卷积**：
    *   卷积层：6 个 $5 \times 5$ 核，`padding=2`（使输出维持 $28 \times 28$）。
    *   激活：Sigmoid（当时 ReLU 还没流行）。
    *   汇聚：$2 \times 2$ 平均汇聚，步幅 2。
3.  **第二块卷积**：
    *   卷积层：16 个 $5 \times 5$ 核，无填充。
    *   激活：Sigmoid。
    *   汇聚：$2 \times 2$ 平均汇聚，步幅 2。
4.  **全连接层**：
    *   展平 (Flatten)。
    *   FC1：120 个神经元 + Sigmoid。
    *   FC2：84 个神经元 + Sigmoid。
    *   输出层：10 个神经元。

---

### 2. 构建 LeNet 的尝试

我们将使用 `nn.Sequential` 快速搭建这个经典模型。

> **维度追踪 (Shape Tracking) 规范：**
> 由于卷积层的空间变换（Padding/Stride）会导致输出形状不再直观，为确保架构设计的准确性，本教程要求在每一层后添加**输出形状注释**。
>
> *   **格式**：注明 `(Batch, Channel, Height, Width)`。
> *   **工程要点**：虽然设计时主要关注 `(C, H, W)` 的逻辑变换，但必须时刻意识到模型是**批处理运行**的，代码逻辑（尤其是 `Flatten` 或自定义层）必须严格保留第一维（Batch 维度）。


In [7]:
import torch
from torch import nn, Tensor

def get_lenet_model() -> nn.Sequential:
    """构建经典 LeNet-5 模型。
    
    使用上述描述的结构。
    """
    net = nn.Sequential(
        # 第一块：卷积 + 激活 + 汇聚
        # 输入: (Batch, 1, 28, 28)
        nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding=2), # -> (Batch, 6, 28, 28)
        nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),                              # -> (Batch, 6, 14, 14)

        # 第二块：卷积 + 激活 + 汇聚
        # 输入: (Batch, 6, 14, 14)
        nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5),           # -> (Batch, 16, 10, 10)
        nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),                              # -> (Batch, 16, 5, 5)

        # 第三块: 全连接层
        # 输入: (Batch, 16, 5, 5)
        nn.Flatten(),                                                       # -> (Batch, 400)
        nn.Linear(16 * 5 * 5, 120),                                         # -> (Batch, 120)
        nn.Sigmoid(),
        nn.Linear(120, 84),                                                 # -> (Batch, 84)
        nn.Sigmoid(),
        nn.Linear(84, 10)                                                   # -> (Batch, 10)
    )
    return net

# --- 验证形状流转 ---
X = torch.randn(size=(1, 1, 28, 28))
net = get_lenet_model()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:<15} 输出形状: {X.shape}")

Conv2d          输出形状: torch.Size([1, 6, 28, 28])
Sigmoid         输出形状: torch.Size([1, 6, 28, 28])
AvgPool2d       输出形状: torch.Size([1, 6, 14, 14])
Conv2d          输出形状: torch.Size([1, 16, 10, 10])
Sigmoid         输出形状: torch.Size([1, 16, 10, 10])
AvgPool2d       输出形状: torch.Size([1, 16, 5, 5])
Flatten         输出形状: torch.Size([1, 400])
Linear          输出形状: torch.Size([1, 120])
Sigmoid         输出形状: torch.Size([1, 120])
Linear          输出形状: torch.Size([1, 84])
Sigmoid         输出形状: torch.Size([1, 84])
Linear          输出形状: torch.Size([1, 10])


---

### 3. 训练与 GPU 加速

由于 CNN 的计算量比 MLP 大，我们必须使用 **5.6 节** 学到的 GPU 知识。


In [ ]:
import d2l_utils as d2l
from torch.utils import data
from typing import Callable

# 对 count_correct 的 gpu 优化版
def count_correct_tensor(y_hat: Tensor, y: Tensor) -> Tensor:
    """【高性能版】计算预测正确的数量，返回 Tensor (不触发 CPU 同步)。
    
    Args:
        y_hat: 预测概率分布。
        y: 真实标签。

    Returns:
        Tensor: 这一批次中预测正确的样本总数。
    """
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    
    cmp = y_hat.type(y.dtype) == y
    return cmp.type(y.dtype).sum()

def evaluate_accuracy_gpu(net: Callable[[Tensor], Tensor], data_iter: data.DataLoader, device: torch.device = None) -> float:
    """使用 GPU 计算在指定数据集上模型的准确率。

    优化说明：
    1. 采用高性能 Accumulator 初始化，减少设备探测。
    2. 使用 count_correct_tensor 替代 count_correct，消除循环内的 CPU-GPU 同步。
    
    Args:
        net: 神经网络模型。
        data_iter: 验证集或测试集的迭代器。
        device: 指定运行的设备。如果为 None，则尝试从模型参数中推断。

    Returns:
        float: 计算得到的模型准确率。
    """
    if isinstance(net, nn.Module):
        net.eval() # 设置为评估模式
        # 如果未指定 device，则取模型第一个参数所在的设备
        if not device:
            try:
                device = next(iter(net.parameters())).device
            except StopIteration:
                # 如果模型没有参数
                device = torch.device('cpu')

    # Accumulator 是我们在 d2l_utils 中定义的累加器
    # 注意，在本章节用的 Accumulator 与前面章节的相比增添了 GPU 优化，建议看一下更新。
    # [正确预测数, 样本总数]
    metric = d2l.Accumulator(2, device)

    with torch.no_grad(): # 评估时不需要计算梯度，节省显存和计算资源
        for X, y in data_iter:
            # 将数据搬运到与模型相同的设备
            X, y = X.to(device), y.to(device)

            # 计算正确数并累加
            # int 会自动转换为 Tensor，且效率更高
            metric.add(count_correct_tensor(net(X), y), y.numel())

    # 在最后返回时，由 .item() 触发唯一一次 CPU-GPU 同步
    # 这会将最终结果（预测正确总数 / 样本总数）从 GPU 取回 CPU 并转为 float
    return (metric[0] / metric[1]).item()

def train_lenet(
    net: nn.Module,
    train_iter: data.DataLoader,
    test_iter: data.DataLoader,
    num_epochs: int,
    lr: float,
    device: torch.device
) -> None:
    """训练函数，用以演示 LeNet 的训练过程。"""

    # 1. 初始化权重
    # 由于用的是 Sigmoid(), 因而使用 4.8 的 Xavier 初始化
    def init_weights(m: nn.Module):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    net.apply(init_weights)
    # 2. 搬运到 GPU
    net.to(device)

    optimizer = torch.optim.SGD(net.parameters(), lr)
    loss = nn.CrossEntropyLoss() # 分类问题，考虑交叉熵损失函数

    print(f"在设备 {device} 上启动训练...")

    for epoch in range(num_epochs):
        net.train()
        metric = d2l.Accumulator(3, device=device) # [train_loss_sum, train_acc_sum, n]

        for X, y in train_iter:
            optimizer.zero_grad()
            # 将数据搬运到 GPU
            X, y = X.to(device), y.to(device)
            y_hat = net(X)

            l = loss(y_hat, y)
            l.backward()
            optimizer.step()

            with torch.no_grad():
                # 传入的是 (Tensor, Tensor, int)，Accumulator 会自动保持在 GPU 上
                metric.add(l * X.shape[0], count_correct_tensor(y_hat, y), X.shape[0])

        # 评估测试集
        train_loss = (metric[0]/metric[2]).item()
        train_acc = (metric[1]/metric[2]).item()
        test_acc = evaluate_accuracy_gpu(net, test_iter, device)
        print(f"--- Epoch {epoch+1} ---")
        print(f"Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}")
        
# 运行示例
batch_size = 256
train_iter, test_iter = d2l.load_fashion_mnist(batch_size=batch_size)

lr, num_epochs = 0.9, 10
device = d2l.get_default_device()
net = get_lenet_model().to(device)

train_lenet(net, train_iter, test_iter, num_epochs, lr, device)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
在设备 cuda 上启动训练...
--- Epoch 1 ---
Loss: 2.317 | Train Acc: 0.101 | Test Acc: 0.100
--- Epoch 2 ---
Loss: 1.474 | Train Acc: 0.421 | Test Acc: 0.579
--- Epoch 3 ---
Loss: 0.860 | Train Acc: 0.660 | Test Acc: 0.699
--- Epoch 4 ---
Loss: 0.704 | Train Acc: 0.725 | Test Acc: 0.711
--- Epoch 5 ---
Loss: 0.631 | Train Acc: 0.753 | Test Acc: 0.753
--- Epoch 6 ---
Loss: 0.580 | Train Acc: 0.775 | Test Acc: 0.737
--- Epoch 7 ---
Loss: 0.535 | Train Acc: 0.793 | Test Acc: 0.759
--- Epoch 8 ---
Loss: 0.503 | Train Acc: 0.809 | Test Acc: 0.816
--- Epoch 9 ---
Loss: 0.479 | Train Acc: 0.820 | Test Acc: 0.804
--- Epoch 10 ---
Loss: 0.457 | Train Acc: 0.831 | Test Acc: 0.817


---

### 4. 细致梳理：LeNet 的关键创新点

1.  **参数共享**：在 $28 \times 28$ 的图像上，6 个 $5 \times 5$ 的核总共只有 $6 \times (25+1) = 156$ 个参数。如果用全连接层，参数量会是数十万。
2.  **空间下采样**：通过平均汇聚层，模型学会了忽略特征的精确位置。这使得模型对图像的轻微旋转和变形更加鲁棒。
3.  **多通道特征提取**：从 1 个输入通道变为 6 个再变为 16 个，通道数在增加，而空间分辨率在减小。**“越深越窄”** 是现代 CNN 的典型特征。

---

### 5. 关键工程暗知识 (Engineering Wisdom)

*   **激活函数的变迁**：原版 LeNet 使用 Sigmoid。在实战中，如果你把 Sigmoid 换成 **ReLU**，你会发现收敛速度快得多，准确率也会提升。
*   **汇聚层的变迁**：原版使用平均汇聚（AvgPool）。现代架构更多使用 **最大汇聚（MaxPool）**，因为它能更显著地提取纹理特征。
*   **计算瓶颈**：你会发现卷积层虽然参数少，但**计算量（FLOPs）**很大；全连接层虽然计算快，但**参数量**巨大，容易导致过拟合。

---
